# 🐳 Clase 3: Orquestación y Operaciones Avanzadas con Docker

---

> **Objetivo de la clase:** Dominar la orquestación de contenedores con Docker Swarm, dar los primeros pasos en Kubernetes desde una perspectiva Docker, implementar un stack completo de observabilidad, gestionar un registro privado de imágenes y aplicar técnicas avanzadas de debugging y performance en entornos productivos.

---

## Tabla de Contenidos

1. [Orquestación con Docker Swarm](#1-orquestación-con-docker-swarm)
2. [Introducción a Kubernetes desde Docker](#2-introducción-a-kubernetes-desde-docker)
3. [Monitoreo Avanzado y Logging](#3-monitoreo-avanzado-y-logging)
4. [Registro Privado y Distribución de Imágenes](#4-registro-privado-y-distribución-de-imágenes)
5. [Performance y Debugging Avanzado](#5-performance-y-debugging-avanzado)
6. [Referencias](#6-referencias)

---

## 1. Orquestación con Docker Swarm

### 🔴 Bloque 1 — Teórico-Práctico

---

### 1.1 Fundamentos de la Orquestación de Contenedores

#### ¿Qué problema resuelve la orquestación?

Ejecutar un único contenedor en una sola máquina es suficiente durante el desarrollo, pero en producción surgen necesidades que Docker standalone no puede cubrir:

```
Problemas en producción sin orquestación:
┌──────────────────────────────────────────────────────────────┐
│  ❌ Un contenedor caído no se reinicia automáticamente       │
│  ❌ No se puede distribuir carga entre múltiples nodos       │
│  ❌ Actualizar la app implica tiempo de inactividad          │
│  ❌ No hay forma nativa de escalar según demanda             │
│  ❌ Los secretos no se distribuyen de forma segura           │
└──────────────────────────────────────────────────────────────┘
```

Un **orquestador de contenedores** resuelve estos problemas al gestionar automáticamente el ciclo de vida de los contenedores en un clúster de nodos.

#### Capacidades clave de la orquestación

| Capacidad | Descripción |
|-----------|-------------|
| **Alta disponibilidad** | Si un nodo falla, los contenedores se reprograman en nodos sanos |
| **Escalado automático** | Aumenta o reduce réplicas según carga o métricas |
| **Self-healing** | Detecta contenedores caídos y los reinicia sin intervención |
| **Rolling updates** | Actualiza servicios sin tiempo de inactividad |
| **Gestión de secretos** | Distribuye credenciales de forma cifrada a los contenedores |
| **Service discovery** | Los servicios se encuentran entre sí por nombre, sin IPs fijas |
| **Load balancing** | Distribuye el tráfico entrante entre réplicas del servicio |

#### Comparativa de orquestadores

```
┌─────────────────┬──────────────────┬──────────────────┬──────────────────┐
│                 │  Docker Swarm    │   Kubernetes     │     Nomad        │
├─────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ Complejidad     │ ⭐ Baja          │ ⭐⭐⭐ Alta      │ ⭐⭐ Media       │
│ Curva de apren. │ ⭐ Suave         │ ⭐⭐⭐ Empinada  │ ⭐⭐ Moderada    │
│ Escalabilidad   │ ⭐⭐ Miles nodos │ ⭐⭐⭐ 5K+ nodos │ ⭐⭐⭐ 10K+ nodos│
│ Ecosistema      │ ⭐ Básico        │ ⭐⭐⭐ Muy rico  │ ⭐⭐ Moderado    │
│ Self-hosted     │ ✅              │ ✅               │ ✅              │
│ Managed cloud   │ ❌              │ ✅ (EKS/GKE/AKS) │ ✅ (HCP Nomad)  │
│ Estado actual   │ Estable/legado  │ Estándar de facto│ En crecimiento  │
└─────────────────┴──────────────────┴──────────────────┴──────────────────┘
```

**¿Cuándo elegir Swarm?**
- Equipo pequeño que ya conoce Docker Compose
- Infraestructura on-premise sencilla (< 50 nodos)
- Migración gradual desde Compose a producción
- Recursos limitados para operar Kubernetes

**¿Cuándo elegir Kubernetes?**
- Escala grande o crecimiento rápido previsto
- Necesitas ecosistema rico (Helm, Istio, ArgoCD, etc.)
- Equipo dedicado a DevOps/Platform Engineering
- Multicloud o híbrido entre cloud y on-premise

---

### 1.2 Docker Swarm en Profundidad

#### Arquitectura de un clúster Swarm

```
┌─────────────────────────────────────────────────────────┐
│                   DOCKER SWARM CLUSTER                  │
│                                                         │
│   ┌─────────────────────────────────────────────────┐   │
│   │              MANAGER NODES                      │   │
│   │  ┌────────────┐  ┌────────────┐  ┌────────────┐ │   │
│   │  │  Manager 1 │  │  Manager 2 │  │  Manager 3 │ │   │
│   │  │  (Leader)  │  │ (Follower) │  │ (Follower) │ │   │
│   │  │            │  │            │  │            │ │   │
│   │  │ Raft Consensus (quórum: mínimo (N/2)+1)    │ │   │
│   │  └─────┬──────┘  └────────────┘  └────────────┘ │   │
│   └────────│────────────────────────────────────────┘   │
│            │ Programación de tareas                      │
│   ┌────────┼────────────────────────────────────────┐   │
│   │        │       WORKER NODES                     │   │
│   │  ┌─────▼──────┐  ┌────────────┐  ┌────────────┐ │   │
│   │  │  Worker 1  │  │  Worker 2  │  │  Worker 3  │ │   │
│   │  │ [replica1] │  │ [replica2] │  │ [replica3] │ │   │
│   │  │ [replica4] │  │ [replica5] │  │            │ │   │
│   │  └────────────┘  └────────────┘  └────────────┘ │   │
│   └────────────────────────────────────────────────┘   │
│                                                         │
│   Overlay Network: todos los nodos se comunican         │
│   de forma cifrada entre sí                             │
└─────────────────────────────────────────────────────────┘
```

**Componentes clave:**
- **Manager nodes:** Mantienen el estado del clúster mediante el algoritmo de consenso Raft. Solo un manager actúa como *Leader* en cada momento.
- **Worker nodes:** Ejecutan las tareas (contenedores) asignadas por el manager. No participan en decisiones del clúster.
- **Quórum:** Para evitar *split-brain*, se necesita mayoría de managers disponibles. Recomendación: 3 o 5 managers.

#### Laboratorio: Inicializar un clúster Swarm

**Paso 1 — Inicializar el nodo manager:**

```bash
# En el nodo que será el manager principal
docker swarm init --advertise-addr <IP_DEL_MANAGER>

# Ejemplo con IP específica:
docker swarm init --advertise-addr 192.168.1.10

# La salida mostrará un token para unir workers, por ejemplo:
# docker swarm join --token SWMTKN-1-xxx... 192.168.1.10:2377
```

**Paso 2 — Obtener tokens para managers y workers:**

```bash
# Token para agregar workers
docker swarm join-token worker

# Token para agregar managers adicionales
docker swarm join-token manager
```

**Paso 3 — Unir nodos workers al clúster:**

```bash
# Ejecutar en cada nodo worker
docker swarm join   --token SWMTKN-1-49nj1cmql0jkz5s954yi3oex3nedyz0fb0xx14ie39trti4wxv-8vxv8rssmk743ojnwacrr2e7c   192.168.1.10:2377
```

**Paso 4 — Verificar el estado del clúster:**

```bash
# Listar nodos del clúster (solo desde un manager)
docker node ls

# Salida esperada:
# ID                            HOSTNAME    STATUS    AVAILABILITY   MANAGER STATUS
# abc123 *                      manager-1   Ready     Active         Leader
# def456                        worker-1    Ready     Active
# ghi789                        worker-2    Ready     Active
```

#### Desplegar servicios con réplicas

```bash
# Crear un servicio con 3 réplicas
docker service create   --name web   --replicas 3   --publish published=80,target=80   nginx:alpine

# Listar servicios
docker service ls

# Ver el estado detallado de un servicio (dónde está cada réplica)
docker service ps web

# Inspeccionar configuración del servicio
docker service inspect --pretty web
```

#### Escalar servicios

```bash
# Escalar a 5 réplicas
docker service scale web=5

# Escalar múltiples servicios a la vez
docker service scale web=5 api=3 worker=2

# Reducir réplicas
docker service scale web=2
```

#### Rolling updates y rollbacks

```bash
# Actualizar la imagen de un servicio con rolling update
docker service update   --image nginx:1.27-alpine   --update-parallelism 2 \       # actualizar 2 réplicas a la vez
  --update-delay 10s \           # esperar 10s entre lotes
  --update-failure-action rollback   web

# Ver historial de actualizaciones
docker service inspect web --pretty | grep -A 10 UpdateConfig

# Hacer rollback manual si algo falla
docker service rollback web
```

#### Stacks con Docker Compose en Swarm

Un **stack** es un grupo de servicios que comparten dependencias y se despliegan juntos, equivalente a `docker-compose` pero para producción en Swarm.

```yaml
# docker-stack.yml
version: "3.9"

services:
  web:
    image: myregistry/webapp:1.0.0
    deploy:
      replicas: 3
      update_config:
        parallelism: 1
        delay: 10s
        failure_action: rollback
      restart_policy:
        condition: on-failure
        max_attempts: 3
      resources:
        limits:
          cpus: "0.50"
          memory: 256M
        reservations:
          cpus: "0.25"
          memory: 128M
    ports:
      - "80:80"
    networks:
      - frontend

  api:
    image: myregistry/api:1.0.0
    deploy:
      replicas: 2
      placement:
        constraints:
          - node.role == worker
    networks:
      - frontend
      - backend
    secrets:
      - db_password

  db:
    image: postgres:16-alpine
    deploy:
      replicas: 1
      placement:
        constraints:
          - node.labels.db == true   # solo en nodos etiquetados
    volumes:
      - db_data:/var/lib/postgresql/data
    networks:
      - backend
    secrets:
      - db_password

networks:
  frontend:
    driver: overlay
  backend:
    driver: overlay
    internal: true      # sin acceso externo

volumes:
  db_data:

secrets:
  db_password:
    external: true      # creado previamente con docker secret create
```

```bash
# Crear el secreto antes del despliegue
echo "SuperSecurePass123!" | docker secret create db_password -

# Desplegar el stack
docker stack deploy -c docker-stack.yml myapp

# Listar stacks
docker stack ls

# Ver servicios del stack
docker stack services myapp

# Ver tareas (contenedores) del stack
docker stack ps myapp

# Eliminar el stack
docker stack rm myapp
```

#### Overlay networks para comunicación entre nodos

```bash
# Crear una red overlay manualmente
docker network create   --driver overlay   --subnet 10.0.9.0/24   --opt encrypted \      # cifrar tráfico entre nodos
  mi-red-overlay

# Los servicios en la misma overlay network se descubren por nombre:
# desde el contenedor 'web' puedo hacer: curl http://api:8080
```

---

### 1.3 Cuándo usar Swarm vs Kubernetes

```
Señales de que Swarm es suficiente:
✅ Equipo ≤ 5 personas en DevOps/infraestructura
✅ Menos de 50 nodos en el clúster
✅ Aplicación con arquitectura de microservicios simple
✅ Necesitas estar en producción rápidamente
✅ Ya tienes experiencia con Docker Compose
✅ No necesitas autoscaling avanzado ni CRDs

Señales de que necesitas Kubernetes:
🔴 Cientos de microservicios con dependencias complejas
🔴 Necesitas Horizontal Pod Autoscaler (HPA)
🔴 Quieres GitOps con ArgoCD o Flux
🔴 Políticas de red granulares (NetworkPolicies)
🔴 Integración con service mesh (Istio, Linkerd)
🔴 Multitenancy con RBAC avanzado
🔴 Ecosistema cloud-native (cert-manager, external-dns, etc.)
```

---

## 2. Introducción a Kubernetes desde Docker

### 🟣 Bloque 2 — Teórico-Práctico

---

### 2.1 Conceptos Clave de Kubernetes para Usuarios de Docker

#### Arquitectura de un clúster Kubernetes

```
┌──────────────────────────────────────────────────────────────────────┐
│                         KUBERNETES CLUSTER                           │
│                                                                      │
│  ┌─────────────────────────────────────────────────────────────────┐ │
│  │                      CONTROL PLANE                              │ │
│  │  ┌────────────────┐  ┌──────────────┐  ┌───────────────────┐   │ │
│  │  │  kube-apiserver│  │    etcd      │  │  kube-scheduler   │   │ │
│  │  │  (API REST)    │  │  (key-value  │  │  (asigna Pods a   │   │ │
│  │  │                │  │   store)     │  │   nodos)          │   │ │
│  │  └────────────────┘  └──────────────┘  └───────────────────┘   │ │
│  │  ┌─────────────────────────────────┐                            │ │
│  │  │  kube-controller-manager        │                            │ │
│  │  │  (reconciliation loop)          │                            │ │
│  │  └─────────────────────────────────┘                            │ │
│  └─────────────────────────────────────────────────────────────────┘ │
│                                                                      │
│  ┌───────────────────────────────────────────────────────────────┐   │
│  │                        WORKER NODES                           │   │
│  │  ┌──────────────────────┐   ┌──────────────────────┐          │   │
│  │  │       Node 1         │   │       Node 2         │          │   │
│  │  │  ┌───┐ ┌───┐ ┌───┐   │   │  ┌───┐ ┌───┐        │          │   │
│  │  │  │Pod│ │Pod│ │Pod│   │   │  │Pod│ │Pod│        │          │   │
│  │  │  └───┘ └───┘ └───┘   │   │  └───┘ └───┘        │          │   │
│  │  │  kubelet  kube-proxy  │   │  kubelet  kube-proxy │          │   │
│  │  └──────────────────────┘   └──────────────────────┘          │   │
│  └───────────────────────────────────────────────────────────────┘   │
└──────────────────────────────────────────────────────────────────────┘
```

#### Equivalencias Docker → Kubernetes

| Concepto Docker | Equivalente Kubernetes | Descripción |
|-----------------|------------------------|-------------|
| `docker run` | **Pod** | Unidad mínima de ejecución (uno o más contenedores) |
| `docker-compose.yml` | **Deployment + Service** | Define el estado deseado y expone el servicio |
| Contenedor | **Container** (dentro de un Pod) | Proceso aislado dentro del Pod |
| `docker network` | **Service** / **NetworkPolicy** | Descubrimiento de servicios y control de tráfico |
| `docker volume` | **PersistentVolume (PV/PVC)** | Almacenamiento persistente desacoplado del Pod |
| `.env` / secrets | **Secret** / **ConfigMap** | Configuración externalizada |
| `docker-compose scale` | **HorizontalPodAutoscaler** | Escala automática basada en métricas |
| `docker stack` | **Helm Chart** / **Kustomize** | Empaquetado y despliegue de aplicaciones completas |
| Docker Hub | **Container Registry** (ECR, GCR, ACR) | Almacén de imágenes |

#### Recursos fundamentales de Kubernetes

**Pod:** La unidad mínima desplegable. Un Pod puede contener uno o más contenedores que comparten red y almacenamiento.

```yaml
# pod.yaml — equivalente a: docker run -p 80:80 nginx:alpine
apiVersion: v1
kind: Pod
metadata:
  name: mi-nginx
  labels:
    app: nginx
spec:
  containers:
    - name: nginx
      image: nginx:alpine
      ports:
        - containerPort: 80
      resources:
        requests:
          memory: "64Mi"
          cpu: "100m"
        limits:
          memory: "128Mi"
          cpu: "200m"
```

**Deployment:** Gestiona un conjunto de Pods replicados y permite rolling updates.

```yaml
# deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: webapp
  namespace: produccion
spec:
  replicas: 3
  selector:
    matchLabels:
      app: webapp
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxUnavailable: 1
      maxSurge: 1
  template:
    metadata:
      labels:
        app: webapp
    spec:
      containers:
        - name: webapp
          image: myregistry/webapp:1.2.0
          ports:
            - containerPort: 8080
          env:
            - name: DB_HOST
              valueFrom:
                secretKeyRef:
                  name: db-secret
                  key: host
          livenessProbe:
            httpGet:
              path: /health
              port: 8080
            initialDelaySeconds: 15
            periodSeconds: 10
          readinessProbe:
            httpGet:
              path: /ready
              port: 8080
            initialDelaySeconds: 5
            periodSeconds: 5
```

**Service:** Expone los Pods internamente o externamente con una IP y DNS estables.

```yaml
# service.yaml
apiVersion: v1
kind: Service
metadata:
  name: webapp-svc
spec:
  selector:
    app: webapp          # selecciona Pods con este label
  ports:
    - protocol: TCP
      port: 80           # puerto del Service
      targetPort: 8080   # puerto del contenedor
  type: ClusterIP        # ClusterIP | NodePort | LoadBalancer
```

---

### 2.2 Entorno Local de Kubernetes

#### Comparativa de opciones para desarrollo local

| Herramienta | Instalación | Recursos | Casos de uso |
|-------------|-------------|----------|--------------|
| **Minikube** | `brew install minikube` | Medio-alto | Aprendizaje, testing local |
| **Kind** | `go install sigs.k8s.io/kind@latest` | Bajo | CI/CD, testing rápido |
| **k3d** | `brew install k3d` | Muy bajo | Desarrollo ágil, multi-clúster local |
| **Docker Desktop** | Activar en Settings | Integrado | Desarrollo en macOS/Windows |

#### Laboratorio: Configurar Kind (Kubernetes in Docker)

Kind ejecuta nodos de Kubernetes como contenedores Docker — no requiere VMs.

```bash
# Instalar Kind
# macOS
brew install kind

# Linux
curl -Lo ./kind https://kind.sigs.k8s.io/dl/v0.23.0/kind-linux-amd64
chmod +x ./kind && sudo mv ./kind /usr/local/bin/kind

# Crear un clúster multi-nodo
cat > kind-cluster.yaml << 'EOF'
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
  - role: control-plane
  - role: worker
  - role: worker
EOF

kind create cluster --name dev --config kind-cluster.yaml

# Verificar que kubectl apunta al clúster Kind
kubectl cluster-info --context kind-dev
kubectl get nodes
```

#### Instalar kubectl

```bash
# macOS
brew install kubectl

# Linux
curl -LO "https://dl.k8s.io/release/$(curl -sSL https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
chmod +x kubectl && sudo mv kubectl /usr/local/bin/

# Verificar
kubectl version --client
```

#### Desplegar una aplicación en Kubernetes

```bash
# Crear namespace para la aplicación
kubectl create namespace demo

# Aplicar manifiestos
kubectl apply -f deployment.yaml -n demo
kubectl apply -f service.yaml -n demo

# Verificar el despliegue
kubectl get pods -n demo -w           # -w: watch (modo live)
kubectl get deployments -n demo
kubectl describe deployment webapp -n demo

# Ver logs de un Pod
kubectl logs -f deployment/webapp -n demo

# Escalar el Deployment
kubectl scale deployment webapp --replicas=5 -n demo

# Actualizar la imagen (rolling update)
kubectl set image deployment/webapp webapp=myregistry/webapp:1.3.0 -n demo

# Ver el estado del rolling update
kubectl rollout status deployment/webapp -n demo

# Rollback si algo falla
kubectl rollout undo deployment/webapp -n demo
```

#### Exponer servicios

```bash
# Port-forward para acceso local temporal (ideal para testing)
kubectl port-forward service/webapp-svc 8080:80 -n demo
# Ahora accesible en http://localhost:8080

# NodePort: accesible en la IP del nodo + puerto alto
kubectl expose deployment webapp --type=NodePort --port=80 -n demo

# LoadBalancer (en cloud o con MetalLB on-premise)
kubectl expose deployment webapp --type=LoadBalancer --port=80 -n demo
kubectl get service webapp -n demo   # ver EXTERNAL-IP asignada
```

---

### 2.3 Kompose: Migrar de Docker Compose a Kubernetes

**Kompose** convierte automáticamente archivos `docker-compose.yml` a manifiestos de Kubernetes, acelerando la migración.

#### Instalación

```bash
# macOS
brew install kompose

# Linux
curl -L https://github.com/kubernetes/kompose/releases/download/v1.33.0/kompose-linux-amd64   -o kompose && chmod +x kompose && sudo mv kompose /usr/local/bin/
```

#### Flujo de migración

```bash
# Tenemos este docker-compose.yml:
# services:
#   web:  image: nginx, ports: ["80:80"]
#   api:  image: myapi:1.0, ports: ["8080:8080"]

# Paso 1 — Convertir a manifiestos
kompose convert -f docker-compose.yml -o k8s/

# Archivos generados en k8s/:
# web-deployment.yaml
# web-service.yaml
# api-deployment.yaml
# api-service.yaml

# Paso 2 — Revisar y ajustar los manifiestos generados
# Kompose puede generar configuraciones no óptimas; revisar:
# - resources (limits/requests no generados automáticamente)
# - liveness/readiness probes
# - PersistentVolumeClaims para volúmenes

# Paso 3 — Aplicar en Kubernetes
kubectl apply -f k8s/ -n mi-namespace

# También se puede desplegar directamente sin generar archivos:
kompose up -f docker-compose.yml

# Eliminar recursos creados por Kompose
kompose down -f docker-compose.yml
```

#### Ajustes típicos post-conversión

```yaml
# ANTES (generado por Kompose — básico)
spec:
  containers:
    - name: api
      image: myapi:1.0

# DESPUÉS (producción — mejorado manualmente)
spec:
  containers:
    - name: api
      image: myapi:1.0
      resources:
        requests:
          memory: "128Mi"
          cpu: "100m"
        limits:
          memory: "256Mi"
          cpu: "500m"
      livenessProbe:
        httpGet:
          path: /health
          port: 8080
        initialDelaySeconds: 10
        periodSeconds: 15
      readinessProbe:
        httpGet:
          path: /ready
          port: 8080
        initialDelaySeconds: 5
        periodSeconds: 5
```

---

## 3. Monitoreo Avanzado y Logging

### 🟠 Bloque 3 — Teórico-Práctico

---

### 3.1 Stack de Observabilidad con Docker Compose

#### Los tres pilares de la observabilidad

```
┌──────────────────────────────────────────────────────────┐
│               OBSERVABILIDAD EN CONTENEDORES             │
│                                                          │
│   ┌──────────────┐  ┌──────────────┐  ┌──────────────┐  │
│   │   MÉTRICAS   │  │    TRAZAS    │  │     LOGS     │  │
│   │              │  │              │  │              │  │
│   │  Prometheus  │  │   Jaeger /   │  │  Loki /      │  │
│   │  + Grafana   │  │   Tempo      │  │  EFK Stack   │  │
│   │              │  │              │  │              │  │
│   │ ¿Qué pasa?   │  │ ¿Por qué?    │  │ ¿Qué dijo?   │  │
│   └──────────────┘  └──────────────┘  └──────────────┘  │
└──────────────────────────────────────────────────────────┘
```

#### Stack completo: Prometheus + Grafana + Loki

```yaml
# docker-compose.monitoring.yml
version: "3.9"

services:

  # ── Métricas de contenedores ──────────────────────────────
  cadvisor:
    image: gcr.io/cadvisor/cadvisor:v0.49.1
    container_name: cadvisor
    volumes:
      - /:/rootfs:ro
      - /var/run:/var/run:ro
      - /sys:/sys:ro
      - /var/lib/docker/:/var/lib/docker:ro
      - /dev/disk/:/dev/disk:ro
    privileged: true
    devices:
      - /dev/kmsg
    networks:
      - monitoring
    restart: unless-stopped

  # ── Scraping y almacenamiento de métricas ────────────────
  prometheus:
    image: prom/prometheus:v2.51.2
    container_name: prometheus
    volumes:
      - ./prometheus/prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - prometheus_data:/prometheus
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"
      - "--storage.tsdb.path=/prometheus"
      - "--storage.tsdb.retention.time=30d"
      - "--web.enable-lifecycle"
    ports:
      - "9090:9090"
    networks:
      - monitoring
    depends_on:
      - cadvisor
    restart: unless-stopped

  # ── Visualización y alertas ──────────────────────────────
  grafana:
    image: grafana/grafana:10.4.2
    container_name: grafana
    volumes:
      - grafana_data:/var/lib/grafana
      - ./grafana/provisioning:/etc/grafana/provisioning:ro
    environment:
      GF_SECURITY_ADMIN_PASSWORD: "${GRAFANA_PASSWORD:-admin}"
      GF_USERS_ALLOW_SIGN_UP: "false"
      GF_FEATURE_TOGGLES_ENABLE: "traceqlEditor"
    ports:
      - "3000:3000"
    networks:
      - monitoring
    depends_on:
      - prometheus
      - loki
    restart: unless-stopped

  # ── Agregación de logs ───────────────────────────────────
  loki:
    image: grafana/loki:3.0.0
    container_name: loki
    volumes:
      - ./loki/loki-config.yml:/etc/loki/local-config.yaml:ro
      - loki_data:/loki
    command: -config.file=/etc/loki/local-config.yaml
    ports:
      - "3100:3100"
    networks:
      - monitoring
    restart: unless-stopped

  # ── Agente de recolección de logs ─────────────────────────
  promtail:
    image: grafana/promtail:3.0.0
    container_name: promtail
    volumes:
      - /var/log:/var/log:ro
      - /var/lib/docker/containers:/var/lib/docker/containers:ro
      - ./promtail/promtail-config.yml:/etc/promtail/config.yml:ro
    command: -config.file=/etc/promtail/config.yml
    networks:
      - monitoring
    depends_on:
      - loki
    restart: unless-stopped

networks:
  monitoring:
    driver: bridge

volumes:
  prometheus_data:
  grafana_data:
  loki_data:
```

#### Configuración de Prometheus

```yaml
# prometheus/prometheus.yml
global:
  scrape_interval: 15s
  evaluation_interval: 15s

scrape_configs:
  # Métricas del propio Prometheus
  - job_name: "prometheus"
    static_configs:
      - targets: ["localhost:9090"]

  # Métricas de contenedores Docker via cAdvisor
  - job_name: "cadvisor"
    static_configs:
      - targets: ["cadvisor:8080"]

  # Métricas del host (si usas node_exporter)
  - job_name: "node"
    static_configs:
      - targets: ["node-exporter:9100"]

  # Descubrimiento automático de contenedores Docker
  - job_name: "docker"
    docker_sd_configs:
      - host: unix:///var/run/docker.sock
    relabel_configs:
      - source_labels: [__meta_docker_container_label_com_docker_compose_service]
        target_label: service
```

#### Configuración de Promtail (recolector de logs)

```yaml
# promtail/promtail-config.yml
server:
  http_listen_port: 9080
  grpc_listen_port: 0

positions:
  filename: /tmp/positions.yaml

clients:
  - url: http://loki:3100/loki/api/v1/push

scrape_configs:
  # Logs de contenedores Docker
  - job_name: docker
    static_configs:
      - targets:
          - localhost
        labels:
          job: docker
          __path__: /var/lib/docker/containers/*/*-json.log
    pipeline_stages:
      - json:
          expressions:
            log: log
            stream: stream
            time: time
      - labels:
          stream:
      - timestamp:
          source: time
          format: RFC3339Nano
```

#### Queries útiles en Grafana (PromQL)

```promql
# CPU usage por contenedor (%)
rate(container_cpu_usage_seconds_total{name!=""}[5m]) * 100

# Memoria usada por contenedor (MB)
container_memory_usage_bytes{name!=""} / 1024 / 1024

# Contenedores reiniciados en las últimas 24h
increase(container_start_time_seconds{name!=""}[24h])

# Red: bytes recibidos por contenedor
rate(container_network_receive_bytes_total[5m])

# Alertas: contenedores con memoria > 80% del límite
(container_memory_usage_bytes / container_spec_memory_limit_bytes) > 0.8
```

#### Crear alertas en Grafana

```yaml
# grafana/provisioning/alerting/rules.yml
apiVersion: 1
groups:
  - orgId: 1
    name: Docker Containers
    folder: Alertas Docker
    interval: 1m
    rules:
      - uid: container-memory-high
        title: "Memoria de contenedor alta"
        condition: C
        data:
          - refId: A
            queryType: ""
            relativeTimeRange:
              from: 300
              to: 0
            datasourceUid: prometheus
            model:
              expr: >
                (container_memory_usage_bytes{name!=""} /
                 container_spec_memory_limit_bytes{name!=""}) > 0.8
              instant: true
        noDataState: NoData
        execErrState: Alerting
        for: 2m
        labels:
          severity: warning
        annotations:
          summary: "Contenedor {{ $labels.name }} usa más del 80% de memoria"
```

---

### 3.2 Logging Centralizado con EFK Stack

El stack **EFK** (Elasticsearch + Fluentd + Kibana) es la alternativa más utilizada en entornos empresariales.

#### Arquitectura del stack EFK

```
Contenedores Docker
      │
      │ stdout/stderr
      ▼
  ┌────────┐    ┌───────────────┐    ┌──────────┐    ┌────────┐
  │ Docker │───▶│   Fluentd     │───▶│ Elastic- │───▶│ Kibana │
  │  Logs  │    │ (recolector + │    │  search  │    │  (UI)  │
  │        │    │  procesador)  │    │ (índice) │    │        │
  └────────┘    └───────────────┘    └──────────┘    └────────┘
```

```yaml
# docker-compose.efk.yml
version: "3.9"

services:
  elasticsearch:
    image: docker.elastic.co/elasticsearch/elasticsearch:8.13.0
    container_name: elasticsearch
    environment:
      - discovery.type=single-node
      - xpack.security.enabled=false
      - "ES_JAVA_OPTS=-Xms512m -Xmx512m"
    volumes:
      - es_data:/usr/share/elasticsearch/data
    ports:
      - "9200:9200"
    networks:
      - efk
    ulimits:
      memlock:
        soft: -1
        hard: -1
    healthcheck:
      test: ["CMD-SHELL", "curl -sf http://localhost:9200/_cluster/health || exit 1"]
      interval: 30s
      timeout: 10s
      retries: 5

  fluentd:
    build: ./fluentd
    container_name: fluentd
    volumes:
      - ./fluentd/conf:/fluentd/etc:ro
    ports:
      - "24224:24224"
      - "24224:24224/udp"
    networks:
      - efk
    depends_on:
      elasticsearch:
        condition: service_healthy

  kibana:
    image: docker.elastic.co/kibana/kibana:8.13.0
    container_name: kibana
    environment:
      ELASTICSEARCH_HOSTS: "http://elasticsearch:9200"
    ports:
      - "5601:5601"
    networks:
      - efk
    depends_on:
      elasticsearch:
        condition: service_healthy

  # App de ejemplo que envía logs a Fluentd
  app:
    image: nginx:alpine
    logging:
      driver: fluentd
      options:
        fluentd-address: localhost:24224
        tag: nginx.access
    networks:
      - efk

networks:
  efk:
    driver: bridge

volumes:
  es_data:
```

#### Dockerfile para Fluentd con plugin Elasticsearch

```dockerfile
# fluentd/Dockerfile
FROM fluent/fluentd:v1.16-debian-1

USER root
RUN gem install fluent-plugin-elasticsearch --no-document
USER fluent
```

#### Configuración de Fluentd

```
# fluentd/conf/fluent.conf
<source>
  @type forward
  port 24224
  bind 0.0.0.0
</source>

# Parsear logs de nginx
<filter nginx.**>
  @type parser
  key_name log
  <parse>
    @type nginx
  </parse>
</filter>

# Enriquecer con metadatos Docker
<filter **>
  @type record_transformer
  <record>
    hostname "#{Socket.gethostname}"
    tag ${tag}
  </record>
</filter>

# Enviar a Elasticsearch
<match **>
  @type elasticsearch
  host elasticsearch
  port 9200
  logstash_format true
  logstash_prefix docker
  include_tag_key true
  tag_key @log_name
  flush_interval 5s
  <buffer>
    @type memory
    flush_interval 5s
    chunk_limit_size 2m
    queue_limit_length 32
    retry_max_interval 30
    retry_forever true
  </buffer>
</match>
```

```bash
# Iniciar el stack EFK
docker compose -f docker-compose.efk.yml up -d

# Verificar que Elasticsearch está sano
curl http://localhost:9200/_cluster/health?pretty

# Ver índices creados por Fluentd
curl http://localhost:9200/_cat/indices?v

# Acceder a Kibana
# Abrir http://localhost:5601
# → Stack Management → Index Patterns → Crear patrón "docker-*"
```

---

## 4. Registro Privado y Distribución de Imágenes

### 🟡 Bloque 4 — Teórico-Práctico

---

### 4.1 Desplegar un Registro Privado

#### ¿Por qué un registro privado?

```
Motivos para usar un registro privado:
┌──────────────────────────────────────────────────────────────┐
│  🔒 Seguridad: imágenes internas no deben ser públicas       │
│  ⚡ Velocidad: pull desde red interna vs internet            │
│  💰 Costo: evitar límites de rate en Docker Hub              │
│  🔍 Control: escaneo de vulnerabilidades propio              │
│  📜 Compliance: requisitos de datos en territorio local       │
└──────────────────────────────────────────────────────────────┘
```

#### Opción 1: Registro básico con `registry:2`

```yaml
# docker-compose.registry.yml
version: "3.9"

services:
  registry:
    image: registry:2
    container_name: registry-privado
    restart: unless-stopped
    ports:
      - "5000:5000"
    environment:
      REGISTRY_HTTP_ADDR: "0.0.0.0:5000"
      REGISTRY_STORAGE_FILESYSTEM_ROOTDIRECTORY: /var/lib/registry
      # Autenticación básica (htpasswd)
      REGISTRY_AUTH: htpasswd
      REGISTRY_AUTH_HTPASSWD_REALM: "Registry Realm"
      REGISTRY_AUTH_HTPASSWD_PATH: /auth/htpasswd
      # TLS (recomendado en producción)
      REGISTRY_HTTP_TLS_CERTIFICATE: /certs/domain.crt
      REGISTRY_HTTP_TLS_KEY: /certs/domain.key
    volumes:
      - registry_data:/var/lib/registry
      - ./auth:/auth:ro
      - ./certs:/certs:ro

volumes:
  registry_data:
```

```bash
# Crear credenciales de autenticación
mkdir -p auth
docker run --rm   --entrypoint htpasswd   httpd:2 -Bbn admin "MySecurePassword" > auth/htpasswd

# Generar certificado autofirmado (solo desarrollo)
mkdir -p certs
openssl req -newkey rsa:4096 -nodes -sha256   -keyout certs/domain.key   -x509 -days 365   -out certs/domain.crt   -subj "/CN=registry.local"

# Iniciar el registro
docker compose -f docker-compose.registry.yml up -d

# Hacer login al registro privado
docker login registry.local:5000
# Usuario: admin, Contraseña: MySecurePassword

# Etiquetar y publicar una imagen al registro privado
docker tag nginx:alpine registry.local:5000/mi-nginx:1.0
docker push registry.local:5000/mi-nginx:1.0

# Descargar desde el registro privado
docker pull registry.local:5000/mi-nginx:1.0

# Listar imágenes en el registro (API REST)
curl -u admin:MySecurePassword   https://registry.local:5000/v2/_catalog

# Ver tags de una imagen específica
curl -u admin:MySecurePassword   https://registry.local:5000/v2/mi-nginx/tags/list
```

#### Opción 2: Harbor — Registro empresarial open source

**Harbor** es el estándar de la industria para registros privados en entornos empresariales. Ofrece:

```
Características de Harbor:
┌────────────────────────────────────────────────────────────┐
│  🔐 RBAC (control de acceso por proyecto y usuario)        │
│  🔍 Escaneo de vulnerabilidades (Trivy integrado)          │
│  🔏 Firma de imágenes con Notary                           │
│  🔄 Replicación entre registros (geo-replication)          │
│  🗑️  Políticas de retención y garbage collection           │
│  📊 Auditoría completa de accesos y operaciones            │
│  🌐 Proxy cache de Docker Hub (evita rate limits)          │
└────────────────────────────────────────────────────────────┘
```

```bash
# Instalar Harbor usando Helm (en Kubernetes)
helm repo add harbor https://helm.goharbor.io
helm install harbor harbor/harbor   --namespace harbor   --create-namespace   --set expose.type=LoadBalancer   --set expose.tls.enabled=true   --set persistence.enabled=true   --set harborAdminPassword="Harbor12345"

# O con Docker Compose (para on-premise):
# Descargar el instalador oficial:
curl -LO https://github.com/goharbor/harbor/releases/download/v2.10.2/harbor-online-installer-v2.10.2.tgz
tar xzvf harbor-online-installer-v2.10.2.tgz
cd harbor

# Configurar harbor.yml (copiar de la plantilla)
cp harbor.yml.tmpl harbor.yml
# Editar: hostname, https.certificate, https.private_key, harbor_admin_password

./install.sh --with-trivy
```

#### Workflow con Harbor

```bash
# Login a Harbor
docker login harbor.empresa.com

# Crear un proyecto en Harbor (via UI o API)
curl -u admin:Harbor12345   -H "Content-Type: application/json"   -X POST https://harbor.empresa.com/api/v2.0/projects   -d '{"project_name":"backend","public":false}'

# Tag y push al proyecto
docker tag myapi:1.0 harbor.empresa.com/backend/myapi:1.0
docker push harbor.empresa.com/backend/myapi:1.0

# Trigger escaneo de vulnerabilidades manual
curl -u admin:Harbor12345   -X POST   "https://harbor.empresa.com/api/v2.0/projects/backend/repositories/myapi/artifacts/1.0/scan"

# Ver resultados del escaneo (resumen)
curl -u admin:Harbor12345   "https://harbor.empresa.com/api/v2.0/projects/backend/repositories/myapi/artifacts/1.0/additions/vulnerabilities"
```

---

### 4.2 Gestión de Imágenes en Producción

#### Estrategia de tagging: Semantic Versioning + SHA

```
Esquema de tags recomendado:
┌──────────────────────────────────────────────────────────────┐
│                                                              │
│  mi-app:1.2.3                  → Versión semántica exacta   │
│  mi-app:1.2                    → Minor version tracking     │
│  mi-app:1                      → Major version tracking     │
│  mi-app:sha-a1b2c3d            → Commit SHA (inmutable)     │
│  mi-app:main-20240101-a1b2c3d  → Branch + fecha + SHA       │
│                                                              │
│  ❌ mi-app:latest              → NUNCA en producción         │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

**¿Por qué no usar `:latest` en producción?**

- `:latest` **cambia** cuando alguien hace `docker push` sin especificar un tag
- No es reproducible: el mismo `docker pull mi-app:latest` puede dar imágenes diferentes en distintos momentos
- Imposible hacer rollback fiable si no sabes qué versión tenías
- Dificulta el debugging y la auditoría

```bash
# Pipeline de tagging recomendado (en CI/CD)
GIT_SHA=$(git rev-parse --short HEAD)
SEMVER="1.2.3"
DATE=$(date +%Y%m%d)

docker build -t mi-app:${SEMVER}              -t mi-app:sha-${GIT_SHA}              -t mi-app:${DATE}-${GIT_SHA}              .

docker push mi-app:${SEMVER}
docker push mi-app:sha-${GIT_SHA}
docker push mi-app:${DATE}-${GIT_SHA}
```

#### Política de retención de imágenes

```bash
# Con Harbor: crear política de retención vía API
curl -u admin:Harbor12345   -H "Content-Type: application/json"   -X POST   "https://harbor.empresa.com/api/v2.0/retentions"   -d '{
    "algorithm": "or",
    "rules": [
      {
        "disabled": false,
        "action": "retain",
        "template": "latestActiveK",
        "params": {"latestActiveK": 10},
        "tag_selectors": [{"kind": "doublestar", "pattern": "**"}],
        "scope_selectors": {"repository": [{"kind": "doublestar", "pattern": "**"}]}
      }
    ],
    "scope": {"level": "project", "ref": 1},
    "trigger": {
      "kind": "Schedule",
      "settings": {"cron": "0 0 0 * * 0"}
    }
  }'

# Limpieza manual del registro básico (registry:2)
# Garbage collection — elimina capas sin referencias
docker exec registry bin/registry garbage-collect   /etc/docker/registry/config.yml --delete-untagged
```

#### Replicación entre registros para Disaster Recovery

```yaml
# Política de replicación en Harbor (via UI: Administration > Replications)
# Configuración equivalente en YAML:
replication_policy:
  name: "replicate-to-dr"
  src_registry: harbor.primary.empresa.com
  dest_registry: harbor.dr.empresa.com
  filters:
    - type: name
      value: "produccion/**"     # solo proyectos de producción
    - type: tag
      value: "v*.*.*"            # solo tags con versión semántica
  trigger:
    type: event_based             # replicar inmediatamente al push
  deletion: false                 # no replicar borrados
  override: true
  enabled: true
```

---

## 5. Performance y Debugging Avanzado

### 🔵 Bloque 5 — Teórico-Práctico

---

### 5.1 Debugging de Contenedores

#### `docker exec` avanzado: adjuntar herramientas de debugging

```bash
# Abrir una shell interactiva en un contenedor en ejecución
docker exec -it <nombre_o_id> bash

# Si el contenedor no tiene bash, usar sh
docker exec -it <nombre_o_id> sh

# Ejecutar un comando puntual sin shell interactiva
docker exec <nombre_o_id> cat /etc/hosts

# Ejecutar como root aunque el contenedor use otro usuario
docker exec -u root -it <nombre_o_id> bash

# Ver variables de entorno del proceso principal
docker exec <nombre_o_id> env

# Ver conexiones de red activas dentro del contenedor
docker exec <nombre_o_id> ss -tunlp
# o (si netstat está disponible):
docker exec <nombre_o_id> netstat -tunlp
```

#### Copiar archivos de/hacia contenedores

```bash
# Copiar archivo DEL contenedor AL host
docker cp mi-contenedor:/app/logs/error.log ./error.log

# Copiar archivo DEL host AL contenedor
docker cp ./config.yml mi-contenedor:/app/config.yml

# Copiar directorio completo
docker cp mi-contenedor:/app/data ./backup_data/

# Útil para extraer logs o binarios generados dentro del contenedor
```

#### Contenedores efímeros para debugging (`docker debug`)

A partir de Docker Desktop 4.27+, el comando `docker debug` permite adjuntar un contenedor efímero con herramientas de diagnóstico a un contenedor existente, **sin necesidad de instalar nada en la imagen original**:

```bash
# Adjuntar un shell de debugging a un contenedor en ejecución
docker debug mi-contenedor

# Esto inicia un contenedor efímero con:
# - shell completa (bash)
# - herramientas: strace, vim, curl, netstat, etc.
# - acceso al filesystem del contenedor objetivo
# El contenedor original NO se ve afectado

# Especificar imagen base para el shell de debugging
docker debug --image=ubuntu:24.04 mi-contenedor
```

#### `nsenter`: acceder al namespace de un contenedor sin Docker

`nsenter` permite entrar directamente a los namespaces del kernel del contenedor — útil cuando Docker no está disponible o para análisis profundo:

```bash
# Obtener el PID del proceso principal del contenedor
CONTAINER_PID=$(docker inspect --format '{{.State.Pid}}' mi-contenedor)

# Entrar al namespace de red del contenedor
nsenter -n -t $CONTAINER_PID -- ss -tunlp

# Entrar al namespace completo (red, PID, mount, etc.)
nsenter -m -u -i -n -p -t $CONTAINER_PID -- bash

# Ver el árbol de procesos dentro del contenedor
nsenter -m -t $CONTAINER_PID -- ps aux
```

#### Analizar un contenedor detenido o con problemas

```bash
# Ver logs aunque el contenedor ya se haya detenido
docker logs --tail 200 mi-contenedor
docker logs --since "10m" mi-contenedor     # últimos 10 minutos

# Inspeccionar el estado detallado del contenedor
docker inspect mi-contenedor

# Ver el código de salida del proceso principal
docker inspect mi-contenedor --format '{{.State.ExitCode}}'
# 0 = éxito, distinto de 0 = error

# Ver los eventos de Docker relacionados con el contenedor
docker events --filter container=mi-contenedor --since "1h"

# Iniciar un contenedor detenido con entrypoint alternativo
docker run --rm -it   --entrypoint bash   mi-imagen:problematica

# Ver diferencias entre el sistema de archivos del contenedor y la imagen base
docker diff mi-contenedor
# A = archivo añadido, C = modificado, D = eliminado
```

---

### 5.2 Performance y Profiling de Contenedores

#### `docker stats`: monitoreo en tiempo real

```bash
# Monitoreo continuo de todos los contenedores
docker stats

# Salida formateada (ideal para scripts)
docker stats --format   "table {{.Name}}	{{.CPUPerc}}	{{.MemUsage}}	{{.NetIO}}	{{.BlockIO}}"

# Captura única (no continua) para scripts
docker stats --no-stream

# Monitorear solo contenedores específicos
docker stats contenedor1 contenedor2

# Ejemplo de salida:
# NAME          CPU %    MEM USAGE / LIMIT     NET I/O       BLOCK I/O
# mi-app        2.45%    128MiB / 256MiB       1.5kB / 2kB   0B / 8.19kB
# postgres      0.10%    89MiB / 512MiB        0B / 0B       2.46MB / 1.84MB
```

#### Inspeccionar uso de recursos con `docker inspect`

```bash
# Ver configuración de recursos del contenedor
docker inspect mi-contenedor --format   '{{.HostConfig.Memory}} bytes memory limit'

docker inspect mi-contenedor --format   '{{.HostConfig.NanoCpus}} nanocpus (divide entre 1e9 para CPUs)'

# Ver todas las estadísticas en tiempo real via API REST
curl --unix-socket /var/run/docker.sock   "http://localhost/containers/mi-contenedor/stats?stream=false" | python3 -m json.tool
```

#### Limitar recursos de contenedores

```bash
# Limitar CPU y memoria al iniciar
docker run -d   --cpus="0.5" \              # máximo 0.5 CPUs (50%)
  --memory="256m" \           # máximo 256 MB RAM
  --memory-swap="512m" \      # límite RAM+swap = 512 MB
  --memory-reservation="128m" \ # reserva "soft" de 128 MB
  --pids-limit=100 \          # máximo 100 procesos
  nginx:alpine

# Actualizar límites de un contenedor en ejecución (sin reiniciarlo)
docker update   --cpus="1.0"   --memory="512m"   mi-contenedor
```

#### Benchmarking de overhead de contenedores

```bash
# Medir tiempo de inicio de un contenedor (cold start)
time docker run --rm alpine echo "ping"

# Benchmark de I/O dentro del contenedor con fio
docker run --rm -it   --volume $(pwd)/testdata:/testdata   ubuntu:22.04 bash -c "
    apt-get install -y fio -q &&
    fio --name=rand-write         --ioengine=libaio         --rw=randwrite         --bs=4k         --numjobs=4         --size=1G         --runtime=30         --filename=/testdata/testfile         --group_reporting
  "

# Stress test de CPU dentro del contenedor
docker run --rm --cpus="1.0" progrium/stress   --cpu 2 --timeout 30s
```

#### Tuning de parámetros del kernel

```bash
# Ver configuración del kernel disponible para contenedores
docker info | grep -i kernel

# sysctl: parámetros de kernel por contenedor (subset seguro)
docker run --rm   --sysctl net.core.somaxconn=65535   --sysctl net.ipv4.tcp_fin_timeout=30   nginx:alpine

# En docker-compose.yml:
# services:
#   nginx:
#     sysctls:
#       net.core.somaxconn: 65535
#       net.ipv4.tcp_fin_timeout: 30

# Aumentar límites de archivos abiertos (ulimits)
docker run --rm   --ulimit nofile=65536:65536   mi-app:latest
```

#### Comparativa de storage drivers

```
Storage Driver   │ Rendimiento │ Estabilidad │ Recomendación
─────────────────┼─────────────┼─────────────┼───────────────────────
overlay2         │ ⭐⭐⭐ Alto  │ ⭐⭐⭐ Muy  │ ✅ Default en Linux
                 │             │    alta      │    (ext4, xfs, btrfs)
─────────────────┼─────────────┼─────────────┼───────────────────────
btrfs            │ ⭐⭐ Medio  │ ⭐⭐ Buena  │ Usar con filesystem
                 │             │              │    btrfs nativo
─────────────────┼─────────────┼─────────────┼───────────────────────
devicemapper     │ ⭐ Bajo     │ ⭐ Requiere │ ❌ Evitar (legacy)
(direct-lvm)     │             │   config.    │
─────────────────┼─────────────┼─────────────┼───────────────────────
zfs              │ ⭐⭐ Medio  │ ⭐⭐⭐ Muy  │ Solo con ZFS nativo
                 │             │    alta      │
─────────────────┼─────────────┼─────────────┼───────────────────────
vfs              │ ⭐ Muy bajo │ ⭐⭐⭐ Alta │ ❌ Solo testing/CI,
                 │             │              │    no producción
```

```bash
# Ver el storage driver actual
docker info | grep "Storage Driver"

# Cambiar a overlay2 (si no es el default)
# Editar /etc/docker/daemon.json:
# {
#   "storage-driver": "overlay2"
# }
sudo systemctl restart docker
```

#### Detectar y resolver fugas de recursos

```bash
# Listar contenedores detenidos que consumen espacio
docker ps -a --filter "status=exited" --format "{{.Names}}: {{.Size}}"

# Ver espacio total usado por Docker
docker system df

# Ver espacio detallado por recurso
docker system df -v

# Limpieza completa del sistema (¡cuidado en producción!)
# Elimina contenedores detenidos, imágenes sin usar, redes y caché de build
docker system prune -a --volumes

# Limpieza selectiva
docker container prune    # solo contenedores detenidos
docker image prune -a     # solo imágenes sin contenedores activos
docker volume prune       # solo volúmenes sin referencias
docker network prune      # solo redes sin contenedores
docker builder prune -a   # solo caché de build
```

---

## 6. Referencias

### 📚 Documentación oficial

| Recurso | URL |
|---------|-----|
| Docker Swarm overview | https://docs.docker.com/engine/swarm/ |
| Docker service create | https://docs.docker.com/engine/reference/commandline/service_create/ |
| Docker Stack deploy | https://docs.docker.com/engine/reference/commandline/stack_deploy/ |
| Kubernetes documentation | https://kubernetes.io/docs/home/ |
| kubectl reference | https://kubernetes.io/docs/reference/kubectl/ |
| Kind (Kubernetes in Docker) | https://kind.sigs.k8s.io/ |
| Minikube docs | https://minikube.sigs.k8s.io/docs/ |
| Kompose | https://kompose.io/ |

### 🔭 Observabilidad y Logging

| Recurso | URL |
|---------|-----|
| Prometheus docs | https://prometheus.io/docs/ |
| Grafana docs | https://grafana.com/docs/ |
| Loki docs | https://grafana.com/docs/loki/latest/ |
| Promtail docs | https://grafana.com/docs/loki/latest/send-data/promtail/ |
| cAdvisor | https://github.com/google/cadvisor |
| Fluentd docs | https://docs.fluentd.org/ |
| Elasticsearch docs | https://www.elastic.co/docs/solutions/observability |
| Kibana docs | https://www.elastic.co/guide/en/kibana/current/ |

### 🗃️ Registros privados

| Recurso | URL |
|---------|-----|
| Docker Registry (registry:2) | https://distribution.github.io/distribution/ |
| Harbor project | https://goharbor.io/ |
| Harbor docs | https://goharbor.io/docs/ |
| Docker Hub rate limits | https://docs.docker.com/docker-hub/download-rate-limit/ |

### 🔧 Performance y Debugging

| Recurso | URL |
|---------|-----|
| docker stats | https://docs.docker.com/engine/reference/commandline/stats/ |
| docker inspect | https://docs.docker.com/engine/reference/commandline/inspect/ |
| docker debug (Desktop 4.27+) | https://docs.docker.com/desktop/use-desktop/container/ |
| Linux namespaces y nsenter | https://man7.org/linux/man-pages/man1/nsenter.1.html |
| Docker storage drivers | https://docs.docker.com/storage/storagedriver/ |
| Resource constraints | https://docs.docker.com/config/containers/resource_constraints/ |

### 📖 Lecturas recomendadas

- **"Docker Deep Dive"** — Nigel Poulton (O'Reilly)
- **"Kubernetes in Action"** — Marko Lukša (Manning)
- **"Cloud Native Patterns"** — Cornelia Davis (Manning)
- **"Observability Engineering"** — Charity Majors, Liz Fong-Jones (O'Reilly)
- Sitio oficial de la CNCF (Cloud Native Computing Foundation): https://www.cncf.io/

---

> 🎓 **Fin de la Clase 3.** Con estos conocimientos, tienes las herramientas para llevar aplicaciones Docker a entornos productivos robustos, observables y escalables. El siguiente paso natural es profundizar en **Kubernetes avanzado** (Helm, GitOps con ArgoCD, service meshes) y en **plataformas cloud-native** como EKS, GKE o AKS.
